# MVP — Mesa de Servicio Inteligente con Alerta Proactiva de Riesgo
Especialización en IA — CUN. Ejecutar en **Google Colab** (*Runtime → Run all*).

Pipeline: **phishing → anonimización (PBD) → sentimiento → clasificación → riesgo → insights → alertas → export para el dashboard**.

> Usa datos sintéticos para correr sin dataset real. Para datos reales, asigna `RUTA`. El índice de riesgo es demostrativo: requiere histórico real de renovación para ser predictivo.

In [ ]:
import pandas as pd, numpy as np, re, json, os, random
from datetime import datetime, timedelta
random.seed(42); np.random.seed(42)
print('Listo')

## 1. Datos (sintéticos o reales)

In [ ]:
RUTA = None   # <-- pon aqui la ruta a tu CSV real; si es None, se generan datos sinteticos

def generar_datos(n=200):
    clientes = [f'Cliente {chr(65+i)}' for i in range(12)]
    base_corr = ['el sistema presenta un error al iniciar sesion','la factura no se genera y muestra falla','se cae la plataforma cada vez que guardo','el boton de pago no responde','error 500 al cargar el reporte']
    base_evol = ['solicito agregar un nuevo campo al formulario','quisiera un reporte adicional por sucursal','necesitamos integrar el modulo con otra app','propongo mejorar el tablero con filtros','solicitud de nueva funcionalidad de exportacion']
    frus = ['urgente','pesimo','inaceptable','molesto','cansado','otra vez']
    phishing = ['estimado usuario verifique su cuenta en http://bit.ly/win haga click o sera suspendida','actualice su contrasena aqui http://secure-login.ru ingrese sus credenciales ahora']
    filas = []
    for i in range(n):
        cli = random.choice(clientes)
        if random.random() < 0.06:
            texto = random.choice(phishing); tipo = 'correctivo'
        else:
            if random.random() < 0.6:
                texto = random.choice(base_corr); tipo = 'correctivo'
            else:
                texto = random.choice(base_evol); tipo = 'evolutivo'
            if random.random() < 0.4:
                texto = random.choice(frus) + ' ' + texto
        creada = datetime(2026,1,1) + timedelta(days=random.randint(0,150))
        horas = round(abs(np.random.normal(24,20)),1)
        filas.append({'id':f'TK-{1000+i}','cliente':cli,'descripcion':texto,'tipo_mantenimiento':tipo,'fecha_creacion':creada,'horas_resolucion':horas})
    return pd.DataFrame(filas)

df = pd.read_csv(RUTA) if RUTA else generar_datos()
print(df.shape)
df.head()

## 2. Seguridad — detección de phishing (aísla y detiene el flujo)

In [ ]:
URLS = re.compile(r'https?://\S+')
SENALES = ['verifique su cuenta','haga click','actualice su contrasena','sera suspendida','ingrese sus credenciales','bit.ly','.ru']

def es_phishing(t):
    t = str(t).lower(); s = 0
    if URLS.search(t): s += 1
    if any(k in t for k in SENALES): s += 2
    return s >= 2

df['phishing'] = df['descripcion'].apply(es_phishing)
aislados = df[df['phishing']]
df_flujo = df[~df['phishing']].copy()
print('Aislados en cuarentena:', len(aislados), '| Continuan al flujo:', len(df_flujo))

## 3. Privacidad — anonimización / enmascaramiento (PBD)

In [ ]:
re_email = re.compile(r'[\w.+-]+@[\w-]+\.[\w.-]+')
re_url   = re.compile(r'https?://\S+')
re_cred  = re.compile(r'(contrasena|password|token|clave)\s*[:=]?\s*\S+', re.I)
re_tel   = re.compile(r'\b\d{7,10}\b')

def anonimizar(t):
    t = str(t)
    t = re_email.sub('[EMAIL]', t)
    t = re_url.sub('[URL]', t)
    t = re_cred.sub('[CRED]', t)
    t = re_tel.sub('[ID]', t)
    return t

df_flujo['texto_anon'] = df_flujo['descripcion'].apply(anonimizar)
df_flujo[['descripcion','texto_anon']].head()

## 4. Análisis de sentimiento (métrica numérica de frustración)
Léxico simple para el MVP. Upgrade sugerido: `pysentimiento` (español).

In [ ]:
POS = set('gracias excelente bien rapido amable solucion resuelto contento satisfecho perfecto'.split())
NEG = set('error falla urgente pesimo inaceptable molesto cansado problema caido lento suspendida'.split())

def sentimiento(t):
    palabras = re.findall(r'[a-zaeioun]+', str(t).lower())
    p = sum(w in POS for w in palabras); n = sum(w in NEG for w in palabras)
    return round((p - n) / (p + n), 3) if (p + n) else 0.0

df_flujo['sentimiento'] = df_flujo['texto_anon'].apply(sentimiento)
df_flujo['frustracion'] = ((1 - df_flujo['sentimiento']) / 2 * 100).round(1)
df_flujo[['sentimiento','frustracion']].describe()

## 5. Clasificación automática (correctivo / evolutivo)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X = df_flujo['texto_anon']; y = df_flujo['tipo_mantenimiento']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
vec = TfidfVectorizer()
clf = LogisticRegression(max_iter=1000).fit(vec.fit_transform(Xtr), ytr)
print(classification_report(yte, clf.predict(vec.transform(Xte))))
df_flujo['clasificacion'] = clf.predict(vec.transform(X))

## 6. Índice de riesgo de no renovación + insights
Variable objetivo **sintética** para el MVP. En datos reales, usar el histórico real de renovación/cancelación.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

df_flujo['fecha_creacion'] = pd.to_datetime(df_flujo['fecha_creacion'])
df_flujo['antiguedad_dias'] = (pd.Timestamp('2026-06-13') - df_flujo['fecha_creacion']).dt.days
agg = df_flujo.groupby('cliente').agg(n_tickets=('id','count'), frus_acum=('frustracion','mean')).reset_index()
df_flujo = df_flujo.merge(agg, on='cliente', how='left')
df_flujo['reincidencia'] = (df_flujo['n_tickets'] > df_flujo['n_tickets'].median()).astype(int)

# Objetivo sintetico (determinista) para que el MVP corra; reemplazar por dato real
prob = (df_flujo['frus_acum']/100*0.5 + df_flujo['antiguedad_dias']/df_flujo['antiguedad_dias'].max()*0.3 + df_flujo['reincidencia']*0.2)
df_flujo['no_renovo'] = (prob > prob.median()).astype(int)

feats = ['frustracion','frus_acum','antiguedad_dias','n_tickets','reincidencia','horas_resolucion']
Xr = df_flujo[feats].fillna(0); yr = df_flujo['no_renovo']
modelo = GradientBoostingClassifier().fit(Xr, yr)
df_flujo['riesgo'] = (modelo.predict_proba(Xr)[:,1] * 100).round(1)

imp = pd.Series(modelo.feature_importances_, index=feats).sort_values(ascending=False)
print('Impulsor principal del riesgo:', imp.index[0])
imp.to_frame('importancia')

## 7. Recomendación y alertas para el account manager

In [ ]:
def banda(r):
    return 'alto' if r >= 66 else ('medio' if r >= 33 else 'bajo')

def recomendacion(r):
    if r >= 66: return 'Contacto inmediato y plan de retencion'
    if r >= 33: return 'Seguimiento proactivo del account manager'
    return 'Monitoreo estandar'

df_flujo['banda_riesgo'] = df_flujo['riesgo'].apply(banda)
df_flujo['recomendacion'] = df_flujo['riesgo'].apply(recomendacion)
alertas = df_flujo[df_flujo['banda_riesgo'] == 'alto'].sort_values('riesgo', ascending=False)
print('Alertas de alto riesgo:', len(alertas))
alertas[['id','cliente','clasificacion','frustracion','riesgo','recomendacion']].head(10)

## 8. Exportar resultados para el dashboard

In [ ]:
bins = [-1, -0.6, -0.2, 0.2, 0.6, 1.0001]
etq = ['muy negativo','negativo','neutro','positivo','muy positivo']
conteo = pd.cut(df_flujo['sentimiento'], bins=bins, labels=etq, include_lowest=True).value_counts().reindex(etq).fillna(0).astype(int)

resultados = {
    'generado': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'kpis': {
        'total_tickets': int(len(df)),
        'phishing_aislados': int(len(aislados)),
        'procesados': int(len(df_flujo)),
        'pct_alto_riesgo': round(float((df_flujo['banda_riesgo']=='alto').mean()*100), 1),
        'frustracion_promedio': round(float(df_flujo['frustracion'].mean()), 1),
        'sentimiento_promedio': round(float(df_flujo['sentimiento'].mean()), 3)
    },
    'clasificacion': {k: int(v) for k, v in df_flujo['clasificacion'].value_counts().items()},
    'distribucion_riesgo': {k: int(v) for k, v in df_flujo['banda_riesgo'].value_counts().reindex(['bajo','medio','alto']).fillna(0).astype(int).items()},
    'sentimiento_hist': {'bins': etq, 'conteo': conteo.tolist()},
    'impulsores': [{'variable': k, 'importancia': round(float(v), 3)} for k, v in imp.items()],
    'tickets_alto_riesgo': [
        {'id': r.id, 'cliente': r.cliente, 'clasificacion': r.clasificacion,
         'frustracion': float(r.frustracion), 'riesgo': float(r.riesgo),
         'impulsor': imp.index[0], 'recomendacion': r.recomendacion}
        for r in alertas.head(15).itertuples()
    ]
}

os.makedirs('docs/data', exist_ok=True)
with open('docs/data/resultados.json', 'w', encoding='utf-8') as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)
print('Guardado en docs/data/resultados.json')
print(json.dumps(resultados['kpis'], indent=2, ensure_ascii=False))

## 9. (Opcional) Descargar el JSON para subirlo al repo
Sube `resultados.json` a `MVP/docs/data/` en GitHub y activa GitHub Pages (Settings → Pages → /docs).

In [ ]:
try:
    from google.colab import files
    files.download('docs/data/resultados.json')
except Exception as e:
    print('Descarga manual del archivo docs/data/resultados.json. Detalle:', e)